In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login


In [3]:
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling
import re

c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento

In [7]:
diretorio = 'G:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\415 - Repositório de Dados\\Repositório Local\\PNAD'

In [9]:
os.chdir(diretorio)  

In [ ]:
os.listdir(diretorio)

In [11]:
df = pd.read_excel('pnad_indicadores_serie_2016_2025.xlsx', sheet_name='indicador_01')
df

,ano,sexo,freq,freq_se,freq_cv,prop,prop_se,prop_cv
0,2016,Homem,4.650154e+06,60894.072647,0.013095,0.438430,0.003871,0.008829
1,2016,Mulher,5.956221e+06,75429.441968,0.012664,0.561570,0.003871,0.006893
2,2017,Homem,4.555177e+06,71551.347587,0.015708,0.434586,0.003788,0.008716
3,2017,Mulher,5.926472e+06,67586.336967,0.011404,0.565414,0.003788,0.006700
4,2018,Homem,4.763627e+06,62369.660775,0.013093,0.439744,0.003912,0.008896
5,2018,Mulher,6.069093e+06,73108.988493,0.012046,0.560256,0.003912,0.006982
6,2019,Homem,4.817930e+06,61934.873475,0.012855,0.438725,0.003897,0.008883
7,2019,Mulher,6.163729e+06,80631.135466,0.013082,0.561275,0.003897,0.006944
8,2020,Homem,4.911042e+06,85462.328913,0.017402,0.438152,0.004740,0.010818
9,2020,Mulher,6.297487e+06,78979.991956,0.012542,0.561848,0.004740,0.008436


In [13]:
df.columns 

Index(['ano', 'sexo', 'freq', 'freq_se', 'freq_cv', 'prop', 'prop_se',
       'prop_cv'],
      dtype='object')

In [14]:
df["freq"] = df["freq"].round().astype("Int64")
df["prop"] = (df["prop"] * 100).round(2)
df = df.drop(columns=['freq_se', 'freq_cv','prop_se', 'prop_cv'])
df = df.rename(columns= {'freq':'quantidade_vinculos', 'sexo':'genero'})   
df = df[['ano','genero','quantidade_vinculos', 'prop']]              
df

,ano,genero,quantidade_vinculos,prop
0,2016,Homem,4650154,43.84
1,2016,Mulher,5956221,56.16
2,2017,Homem,4555177,43.46
3,2017,Mulher,5926472,56.54
4,2018,Homem,4763627,43.97
5,2018,Mulher,6069093,56.03
6,2019,Homem,4817930,43.87
7,2019,Mulher,6163729,56.13
8,2020,Homem,4911042,43.82
9,2020,Mulher,6297487,56.18


In [19]:
def transformar(nome):
    nome =  re.sub("Homem", "Masculino", nome)
    nome = re.sub("Mulher", "Feminino", nome)
    return nome

In [ ]:
df['genero'] = df['genero'].apply(transformar)
df['genero'].unique()

In [21]:
df

,ano,genero,quantidade_vinculos,prop
0,2016,Masculino,4650154,43.84
1,2016,Feminino,5956221,56.16
2,2017,Masculino,4555177,43.46
3,2017,Feminino,5926472,56.54
4,2018,Masculino,4763627,43.97
5,2018,Feminino,6069093,56.03
6,2019,Masculino,4817930,43.87
7,2019,Feminino,6163729,56.13
8,2020,Masculino,4911042,43.82
9,2020,Feminino,6297487,56.18


# Upload

In [18]:
df.columns

Index(['ano', 'genero', 'quantidade_vinculos', 'prop'], dtype='object')

In [24]:
for col in df.columns:
    print(df[col].unique())
    print('-----------------------------')

[2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]
-----------------------------
['Masculino' 'Feminino']
-----------------------------
<IntegerArray>
[4650154, 5956221, 4555177, 5926472, 4763627, 6069093, 4817930, 6163729,
 4911042, 6297487, 4958877, 6249463, 4828278, 6231642, 4899578, 6668135,
 5049868, 6734757, 5162774, 7064334]
Length: 20, dtype: Int64
-----------------------------
[43.84 56.16 43.46 56.54 43.97 56.03 43.87 56.13 43.82 56.18 44.24 55.76
 43.66 56.34 42.36 57.64 42.85 57.15 42.22 57.78]
-----------------------------


In [15]:
client = bigquery.Client(project='repositoriodedadosgpsp')

In [22]:
schema = [bigquery.SchemaField('ano', 'INTEGER', description= 'Ano de referência da observação'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),         
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop', 'FLOAT', description= 'Proporção de vínculos em relação ao total naquele ano'),
          ]

dataset_ref = client.dataset('perfil_remuneracao')

table_ref = dataset_ref.table('PNAD_vinculos_genero') 
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=d47fae60-c2a2-494c-9d0b-6249358d8b9b>